In [23]:
import importlib
importlib.reload(data_prep)

<module 'data_prep' from '/Users/ehabhasan/Documents/Projects/Time Series Forecasting/Large Scale Forecasting/data_prep.py'>

In [ ]:
import pandas as pd
import data_prep  


pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)        # Avoid line wrapping
pd.set_option('display.max_colwidth', None) # Show full content of each cell

# Import and Prepare Electricity Load dataset

In [25]:
# Load hourly electricity data, excluding non-EU countries and those with significant missing values
df=data_prep.load_elec_hourly_data(
    ["Data/monthly_hourly_load_values_2025.csv",
    "Data/monthly_hourly_load_values_2024.csv",
    "Data/monthly_hourly_load_values_2023.csv"],
    ['CY','IE','AL', 'BA', 'CH', 'GB', 'GE', 'MD', 'ME', 'MK', 'NO', 'RS', 'XK'])

- Check for duplicates: Duplicate records were found, possibly due to clock shifts (e.g., daylight saving time).

In [26]:
data_prep.duplicate_check(df,['Value'],True)

                      Date CountryCode  Count
48429  2023-03-26 03:00:00          AT      2
48430  2023-03-26 03:00:00          BE      2
48431  2023-03-26 03:00:00          BG      2
48432  2023-03-26 03:00:00          CZ      2
48433  2023-03-26 03:00:00          DE      2
...                    ...         ...    ...
471647 2025-03-30 03:00:00          PL      2
471648 2025-03-30 03:00:00          PT      2
471650 2025-03-30 03:00:00          SE      2
471651 2025-03-30 03:00:00          SI      2
471652 2025-03-30 03:00:00          SK      2

[69 rows x 3 columns]


'Number of duplicated records 69'

- A solution will take the first occurence

In [27]:
df=df.drop_duplicates(subset=['CountryCode','Date'],keep='first')

In [28]:
data_prep.duplicate_check(df,['Value'],True)

'Number of duplicated records 0'

**Detect Gaps in the hourly time series by country group**
- As we can see, some hourly timestamps are missing from the data.

In [29]:
data_prep.CheckRangePerColumn(df,'CountryCode','Date',False)

- Generate Complete Hourly Time Series per Country from 2023 to 2025

In [30]:
df_full_range=df.copy()
df_full_range = pd.concat(
        [data_prep.full_date_range(df[df["CountryCode"] == country], 'Date', country, freq='h') 
        for country in df["CountryCode"].dropna().unique()],
        ignore_index=True
    )

- As we can see the date range is complete now

In [31]:
data_prep.CheckRangePerColumn(df_full_range,'CountryCode','Date',False)

- Filling in the full hourly timeline per country shows missing data as null values

In [32]:
df_full_range.isnull().sum()

Date             0
CountryCode      0
Value          187
dtype: int64

- Missing values are filled using interpolation (performed per country), which estimates missing data points by leveraging surrounding values to ensure a smooth and continuous time series.

In [33]:
df_full_range['Value']=df_full_range.groupby('CountryCode')['Value'].transform(lambda x: x.interpolate(method='linear'))

- The percentage differences in statistical properties between the original and interpolated datasets are very low across all countries, indicating minimal distortion from interpolation.

In [34]:
data_prep.InterDQ(df,df_full_range,'CountryCode','Value')


,mean_pct_diff,std_pct_diff,min_pct_diff,max_pct_diff
CountryCode,,,,
AT,0.004,0.006,0.0,0.0
BE,0.003,0.006,0.0,0.0
BG,0.004,0.004,0.0,0.0
CZ,0.004,0.006,0.0,0.0
DE,0.004,0.015,0.0,0.0
DK,0.001,0.003,0.0,0.0
EE,0.002,0.006,0.0,0.0
ES,0.004,0.014,0.0,0.0
FI,0.001,0.004,0.0,0.0


# Import and Prepare Weather Data

- Fetch Hourly Weather Data for EU Countries (2023–2025) via Open-Meteo REST API

In [35]:
# Determine the full time range of interest from the dataset
start_date=df_full_range['Date'].min().strftime('%Y-%m-%d')
end_date=df_full_range['Date'].max().strftime('%Y-%m-%d')

# Define a dictionary of EU countries with their approximate latitude and longitude
# These coordinates are used to fetch corresponding weather data
eu_countries = {
    'AT': (47.5162, 14.5501), 'BE': (50.5039, 4.4699), 'BG': (42.7339, 25.4858),
    'CZ': (49.8175, 15.4730), 'DE': (51.1657, 10.4515), 'DK': (56.2639, 9.5018),
    'EE': (58.5953, 25.0136), 'ES': (40.4637, -3.7492), 'FI': (61.9241, 25.7482),
    'FR': (46.6034, 1.8883),  'GR': (39.0742, 21.8243),  'HR': (45.1, 15.2),
    'HU': (47.1625, 19.5033), 'IT': (41.8719, 12.5674),  'LT': (55.1694, 23.8813),
    'LU': (49.8153, 6.1296),  'LV': (56.8796, 24.6032),  'NL': (52.1326, 5.2913),
    'PL': (51.9194, 19.1451), 'PT': (39.3999, -8.2245),  'RO': (45.9432, 24.9668),
    'SE': (60.1282, 18.6435), 'SI': (46.1512, 14.9955),  'SK': (48.6690, 19.6990)
}

# This pulls external data (e.g., temperature, humidity, solar radiation) per location
weather_df=data_prep.fetch_eu_weather(eu_countries=eu_countries, start_date=start_date, end_date=end_date)

In [36]:
weather_df.head()

,Date,Temp_C,Humidity,WindSpeed,SolarRadiation,CloudCover,CountryCode
0,2023-01-01 00:00:00,4.2,77,13.0,0.0,82,AT
1,2023-01-01 01:00:00,3.9,78,11.5,0.0,100,AT
2,2023-01-01 02:00:00,3.7,76,11.6,0.0,100,AT
3,2023-01-01 03:00:00,2.9,78,9.5,0.0,98,AT
4,2023-01-01 04:00:00,1.8,83,6.9,0.0,100,AT


- Quick check to make sure the date range is equal

In [37]:
len(df_full_range['Date'].unique()) - len(weather_df['Date'].unique())

0

# Combine both datasets

In [38]:
data=df_full_range.merge(weather_df,on=['CountryCode','Date'],how='left')
data.head()

,Date,CountryCode,Value,Temp_C,Humidity,WindSpeed,SolarRadiation,CloudCover
0,2023-01-01 00:00:00,AT,5280.8,4.2,77,13.0,0.0,82
1,2023-01-01 01:00:00,AT,5143.0,3.9,78,11.5,0.0,100
2,2023-01-01 02:00:00,AT,4958.7,3.7,76,11.6,0.0,100
3,2023-01-01 03:00:00,AT,4905.4,2.9,78,9.5,0.0,98
4,2023-01-01 04:00:00,AT,5025.8,1.8,83,6.9,0.0,100


- Quick check to make sure nothing broke

In [39]:
data_prep.CheckRangePerColumn(data,'CountryCode','Date',False)

In [40]:
len(df_full_range)-len(data)

0

- save the the df into csv file

In [41]:
# data.to_csv("Data/data.csv", index=False)